# 65 Pairwise inter-landmark distances

Deletion preserves every non-masked vertex bit-exact, so the ten pairwise distances among the five 10-20 landmarks must match between the original mesh and the anonymized mesh. This notebook computes that table for one representative subject (Subject 17 by default) and populates Table 4.3 of the thesis.

The check is reported for one subject because the deletion operator is the same for every subject; the remaining six subjects' entries in Table 4.3 would be identical zeros and add no information.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    SUBJECTS, subject_paths, load_raw, load_anon, load_landmarks,
    available_subjects, missing_report, run_pipeline,
)

import numpy as np
import pandas as pd

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

SUBJECT = 17

## 1. Load original and anonymized landmarks

The TSV written by `save_anonymized_scan` uses the digitized frame, the same frame as the raw OBJ. Both the original and anonymized landmarks therefore live in a common coordinate system; any difference between pairwise distances reflects landmark displacement, not a frame mismatch.

In [ ]:
import cedalion.io

paths = subject_paths(SUBJECT)

landmarks_orig = cedalion.io.load_tsv(str(paths.landmarks_tsv))

print('Original landmarks:')

print(landmarks_orig)

## 2. Re-run pipeline to obtain the anonymized landmarks

`save_anonymized_scan` writes a single `_landmarks.tsv` -- the landmarks as they sit on the anonymized output. For this comparison we want both: the landmarks on the original (just loaded) and the landmarks after the affine round-trip.  Since the deletion step itself does not touch the landmark array, we can simply re-run the pipeline and take `landmarks_dig`.

In [ ]:
surface_raw = load_raw(SUBJECT)

art = run_pipeline(surface_raw, landmarks_orig, subject=SUBJECT)

landmarks_anon = art.landmarks_dig

print('Anonymized landmarks:')

print(landmarks_anon)

## 3. Compute pairwise distances

All 10 unordered pairs among {Nz, Iz, Cz, Lpa, Rpa}.

In [ ]:
from itertools import combinations



def _pos(da, label):

    arr = da.pint.dequantify().values

    labels = list(da['label'].values)

    return arr[labels.index(label)]



labels = ['Nz', 'Iz', 'Cz', 'LPA', 'RPA']

rows = []

for a, b in combinations(labels, 2):

    d_orig = float(np.linalg.norm(_pos(landmarks_orig, a) - _pos(landmarks_orig, b)))

    d_anon = float(np.linalg.norm(_pos(landmarks_anon, a) - _pos(landmarks_anon, b)))

    rows.append({

        'pair': f'{a}-{b}',

        'd_original_mm': d_orig,

        'd_anonymized_mm': d_anon,

        'abs_diff_mm': abs(d_orig - d_anon),

    })



df = pd.DataFrame(rows)

df

## 4. Save

In [ ]:
out = OUT_DIR / 'pairwise_distances.csv'

df.to_csv(out, index=False)

print(f'Max |diff|: {df.abs_diff_mm.max():.6g} mm')

print(f'Wrote {out}')